In [1]:
# Tugas Mandiri

# Sel ini menghasilkan TIGA dataset cabang (kota) terpisah untuk Tugas Mandiri Pertemuan 3
import numpy as np
import pandas as pd

kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-08-01", "2026-08-31", freq="D")

cabang_kota = {"Magelang": 101, "Yogyakarta": 202, "Semarang": 303}

for kota, seed in cabang_kota.items():
    np.random.seed(seed)  # seed berbeda tiap kota agar datanya bervariasi, namun tetap konsisten/reproducible
    n = 200
    data_cabang = {
        "order_id": [f"{kota[:3].upper()}-{2000 + i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25, 0.25, 0.20, 0.15, 0.15]),
        "unit_terjual": np.random.randint(1, 8, size=n),
        "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    }
    df_cabang = pd.DataFrame(data_cabang)
    df_cabang["kota"] = kota
    nama_file = f"transaksi_{kota.lower()}.csv"
    df_cabang.to_csv(nama_file, index=False)
    print(f"Berkas '{nama_file}' berhasil dibuat: {df_cabang.shape[0]} baris")

print("\nKetiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.")

Berkas 'transaksi_magelang.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_yogyakarta.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_semarang.csv' berhasil dibuat: 200 baris

Ketiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.


In [4]:
# A. Membangun Struktur Direktori HDFS

!hdfs dfs -mkdir -p /user/krock/ecommerce/raw
!hdfs dfs -mkdir -p /user/krock/ecommerce/processed


!hdfs dfs -ls -R /user/krock/ecommerce


drwxr-xr-x   - krock supergroup          0 2026-09-09 20:11 /user/krock/ecommerce/processed
drwxr-xr-x   - krock supergroup          0 2026-09-09 20:11 /user/krock/ecommerce/raw


In [10]:
# B. Mengunggah Data Mentah ke HDFS
# Unggah ketiga berkas CSV (transaksi_magelang.csv, transaksi_yogyakarta.csv, transaksi_semarang.csv) ke direktori /user/[username]/ecommerce/raw di HDFS. 
#Tampilkan isi direktori tersebut sebagai bukti ketiga berkas berhasil terunggah, lengkap dengan ukurannya.

# upload berkas
# !hdfs dfs -put transaksi_magelang.csv /user/krock/ecommerce/raw
# !hdfs dfs -put transaksi_yogyakarta.csv /user/krock/ecommerce/raw
# !hdfs dfs -put transaksi_semarang.csv /user/krock/ecommerce/raw

# tampilkan direktori
!hdfs dfs -du -h /user/krock/ecommerce/raw
!hdfs dfs -count /user/krock/ecommerce/raw


12.0 K  12.0 K  /user/krock/ecommerce/raw/transaksi_magelang.csv
11.9 K  11.9 K  /user/krock/ecommerce/raw/transaksi_semarang.csv
12.4 K  12.4 K  /user/krock/ecommerce/raw/transaksi_yogyakarta.csv
           1            3              37167 /user/krock/ecommerce/raw


In [12]:
# C. Membaca Kembali dan Menggabungkan Data dari HDFS

import pandas as pd

# Mengunduh data hdfs agar dibaca oleh pandas
# !hdfs dfs -get /user/krock/ecommerce/raw/transaksi_magelang.csv 
# !hdfs dfs -get /user/krock/ecommerce/raw/transaksi_yogyakarta.csv 
# !hdfs dfs -get /user/krock/ecommerce/raw/transaksi_semarang.csv 

# Membaca ketiga file CSV
df_magelang = pd.read_csv("transaksi_magelang.csv")
df_yogyakarta = pd.read_csv("transaksi_yogyakarta.csv")
df_semarang = pd.read_csv("transaksi_semarang.csv")

# Menggabungkan menjadi satu DataFrame
df_gabungan = pd.concat([df_magelang, df_yogyakarta, df_semarang], ignore_index=True)

# Membuktikan hasil gabungan berisi transaksi dari ketiga kota
print("Distribusi data per kota:")
print(df_gabungan["kota"].value_counts())
df_gabungan.head()


Distribusi data per kota:
kota
Magelang      200
Yogyakarta    200
Semarang      200
Name: count, dtype: int64


,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang


In [15]:
# D. Mengolah dan Download Hasil ke Direktori processed

# Menambahkan kolom total_pendapatan (unit_terjual * harga_satuan)
df_gabungan["total_pendapatan"] = df_gabungan["unit_terjual"] * df_gabungan["harga_satuan"]

# Membuat tabel ringkasan (groupby kota dan kategori)
ringkasan_df = df_gabungan.groupby(["kota", "kategori"]).agg(
    total_unit_terjual=("unit_terjual", "sum"),
    total_pendapatan=("total_pendapatan", "sum"),
    jumlah_transaksi=("order_id", "count")
).reset_index()

# Menyimpan ke CSV lokal
df_gabungan.to_csv("data_gabungan_bersih.csv", index=False)
ringkasan_df.to_csv("ringkasan_kota_kategori.csv", index=False)

# Mengunggah kedua file hasil olahan ke direktori processed di HDFS
!hdfs dfs -put data_gabungan_bersih.csv /user/krock/ecommerce/processed/
!hdfs dfs -put ringkasan_kota_kategori.csv /user/krock/ecommerce/processed/

# Verifikasi file berhasil terunggah di HDFS
!hdfs dfs -ls -R /user/krock/ecommerce/processed

put: `/user/krock/ecommerce/processed/data_gabungan_bersih.csv': File exists
put: `/user/krock/ecommerce/processed/ringkasan_kota_kategori.csv': File exists
-rw-r--r--   1 krock supergroup      41257 2026-09-09 20:50 /user/krock/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 krock supergroup        671 2026-09-09 20:50 /user/krock/ecommerce/processed/ringkasan_kota_kategori.csv


# E. Dokumentasi dan Refleksi

# 1. Screenshot halaman NameNode Web UI (http://localhost:9870, menu Utilities → Browse the file system) yang menunjukkan struktur folder /user/[username]/ecommerce/ beserta isinya (paste gambar ke dalam markdown cell, atau lampirkan terpisah jika format pengumpulan berupa arsip/zip.

![Screenshot HDFS WEB UI](hdfs_ui.png)


# 2. Tulisan reflektif (minimal 100 kata) pada markdown cell yang menjawab: Apa keuntungan menyimpan data mentah (raw) terpisah dari data olahan (processed) di HDFS, dibandingkan menyimpan semuanya bercampur dalam satu folder?

> Menyimpan data mentah (raw) terpisah secara ketat dari data olahan (processed) di dalam HDFS merupakan praktik arsitektur data yang sangat krusial dalam lingkungan Big Data. Pemisahan ini memastikan integritas data asli tetap terjaga tanpa risiko tertimpa atau rusak akibat kesalahan proses transformasi atau manipulasi analitik (immutability principle). Data raw berfungsi sebagai single source of truth yang bersih dan autentik dari sumber aslinya (seperti transaksi cabang), yang sewaktu-waktu dapat diolah ulang dengan logika bisnis yang berbeda di masa depan. Sebaliknya, direktori processed didedikasikan untuk menyimpan hasil turunan yang siap konsumsi bagi tim analyst atau dashboarding, sehingga efisiensi penyimpanan, kecepatan akses query, serta manajemen alur kerja (pipeline data) menjadi jauh lebih terstruktur, transparan, dan mudah diaudit
